Perform polynomial and interaction terms in linear regression. 

Detect multicollinearity using a variance inflation factor or correlation. 

Apply continuous and categorical features in linear regression. 

Design an optimal modeling strategy that selects between polynomial terms, interactive terms, and/or continuous and categorical features based on dataset characteristics.  

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from patsy import dmatrices
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor

1. Apply Continuous and Categorical Features in Linear Regression
First, we load the data, handle missing values, and encode categorical variables using One-Hot Encoding to make them usable in a linear regression model.

In [2]:
# load dataset
df = pd.read_csv('medical_insurance.csv')

# select relevant baseline features (mix of continuous and categorical)
continuous_features = ['age', 'bmi', 'visits_last_year', 'income']
categorical_features = ['sex', 'smoker', 'region']
target = 'annual_medical_cost'

# Drop rows with missing values in our target or features
df_clean = df.dropna(subset=continuous_features + categorical_features + [target])

# Separate features and target
X_raw = df_clean[continuous_features + categorical_features]
y = df_clean[target]

# One-hot encode categorical features (dropping first to avoid perfect multicollinearity)
X_baseline = pd.get_dummies(X_raw, columns=categorical_features, drop_first=True)

# Convert boolean flags to integers (0 or 1)
X_baseline = X_baseline.astype(float)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_baseline, y, test_size=0.2, random_state=42
)

# Fit Baseline Model
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

# Evaluate Baseline
baseline_preds = baseline_model.predict(X_test)
print(f"Baseline Test R²: {r2_score(y_test, baseline_preds):.4f}")
print(
    f"Baseline Test RMSE: {np.sqrt(mean_squared_error(y_test, baseline_preds)):.2f}"
)

Baseline Test R²: 0.0749
Baseline Test RMSE: 3017.20


2. Detect Multicollinearity Using Variance Inflation Factor (VIF)
Before adding complex terms, we check for multicollinearity among our continuous predictors using VIF. A VIF greater than 5 or 10 indicates high multicollinearity, which can destabilize our regression coefficients.

In [3]:
# Create a dataframe to store VIF results for continuous features
vif_data = pd.DataFrame()
vif_data["feature"] = continuous_features

# Add a constant column for the intercept (required by statsmodels VIF calculation)
X_vif_check = sm.add_constant(df_clean[continuous_features])

# Calculate VIF for each feature
vif_data["VIF"] = [
    variance_inflation_factor(X_vif_check.values, i)
    for i in range(1, X_vif_check.shape[1])
]

print("\n--- Variance Inflation Factor (VIF) ---")
print(vif_data.to_string(index=False))

# Optional Rule of Thumb: If VIF > 5, consider dropping or combining features.


--- Variance Inflation Factor (VIF) ---
         feature      VIF
             age 1.003997
             bmi 1.000018
visits_last_year 1.003972
          income 1.000036


3. Perform Polynomial and Interaction Terms
To capture non-linear relationships (like age accelerating costs) and synergies (like bmi having a worse impact if the person is a smoker), we generate polynomial and interaction features.

In [4]:
# Create a copy of the training data to build engineering features safely
X_train_eng = X_train.copy()
X_test_eng = X_test.copy()

# 1. Polynomial Term: Age squared (Non-linear relationship)
X_train_eng["age_squared"] = X_train_eng["age"] ** 2
X_test_eng["age_squared"] = X_test_eng["age"] ** 2

# 2. Interaction Term: BMI * Smoker status (assuming smoker_Yes exists after dummy encoding)
# Note: Check your dummy column name if it varies
smoker_col = [col for col in X_train.columns if "smoker_" in col]

if smoker_col:
    # Use the first encoded smoker column found (e.g., smoker_Yes)
    target_smoker = smoker_col[0]
    X_train_eng["bmi_x_smoker"] = X_train_eng["bmi"] * X_train_eng[target_smoker]
    X_test_eng["bmi_x_smoker"] = X_test_eng["bmi"] * X_test_eng[target_smoker]
    print(
        f"\nSuccessfully added interaction term: bmi x {target_smoker}"
    )

# Fit the Feature Engineered Model
eng_model = LinearRegression()
eng_model.fit(X_train_eng, y_train)

# Evaluate Engineering Model
eng_preds = eng_model.predict(X_test_eng)
print(f"Engineered Model Test R²: {r2_score(y_test, eng_preds):.4f}")
print(
    f"Engineered Model Test RMSE: {np.sqrt(mean_squared_error(y_test, eng_preds)):.2f}"
)


Successfully added interaction term: bmi x smoker_Former
Engineered Model Test R²: 0.0752
Engineered Model Test RMSE: 3016.73


4. Design an Optimal Modeling StrategyAn automated strategy should select features dynamically based on dataset characteristics. Below, we use Adjusted $R^2$ as our decision matrix because it penalizes model complexity, ensuring we only keep polynomial or interaction terms if they genuinely add predictive power.

In [5]:
def optimal_modeling_strategy(X_tr, X_te, y_tr, y_te, cont_feats, cat_feats):
    """Dynamically tests and selects the optimal regression framework

    based on Adjusted R-squared comparison.
    """

    def get_adj_r2(X_train_df, X_test_df, y_train_series, y_test_series):
        model = LinearRegression()
        model.fit(X_train_df, y_train_series)
        preds = model.predict(X_test_df)
        r2 = r2_score(y_test_series, preds)
        n = len(y_test_series)
        p = X_test_df.shape[1]
        # Calculate Adjusted R-squared
        adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))
        return adj_r2, model

    # Strategy 1: Baseline Only
    adj_r2_base, model_base = get_adj_r2(X_tr, X_te, y_tr, y_te)

    # Strategy 2: Include Polynomials (quadratic terms for continuous features)
    X_tr_poly, X_te_poly = X_tr.copy(), X_te.copy()
    for col in cont_feats:
        X_tr_poly[f"{col}_sq"] = X_tr_poly[col] ** 2
        X_te_poly[f"{col}_sq"] = X_te_poly[col] ** 2
    adj_r2_poly, model_poly = get_adj_r2(X_tr_poly, X_te_poly, y_tr, y_te)

    # Strategy 3: Include Interactions (cross-multiplication of continuous features)
    X_tr_inter, X_te_inter = X_tr.copy(), X_te.copy()
    for i in range(len(cont_feats)):
        for j in range(i + 1, len(cont_feats)):
            c1, c2 = cont_feats[i], cont_feats[j]
            X_tr_inter[f"{c1}_x_{c2}"] = X_tr_inter[c1] * X_tr_inter[c2]
            X_te_inter[f"{c1}_x_{c2}"] = X_te_inter[c1] * X_te_inter[c2]
    adj_r2_inter, model_inter = get_adj_r2(X_tr_inter, X_te_inter, y_tr, y_te)

    # Strategy 4: Full Combination (Baseline + Poly + Interaction)
    X_tr_full, X_te_full = X_tr_poly.copy(), X_te_poly.copy()
    for i in range(len(cont_feats)):
        for j in range(i + 1, len(cont_feats)):
            c1, c2 = cont_feats[i], cont_feats[j]
            X_tr_full[f"{c1}_x_{c2}"] = X_tr_full[c1] * X_tr_full[c2]
            X_te_full[f"{c1}_x_{c2}"] = X_te_full[c1] * X_te_full[c2]
    adj_r2_full, model_full = get_adj_r2(X_tr_full, X_te_full, y_tr, y_te)

    # Decision Matrix Evaluation
    results = {
        "Baseline (Continuous + Categorical)": adj_r2_base,
        "Polynomial Strategy": adj_r2_poly,
        "Interaction Strategy": adj_r2_inter,
        "Full Combined Strategy": adj_r2_full,
    }

    best_strategy = max(results, key=results.get)

    print("\n--- Strategy Selection Performance (Adjusted R²) ---")
    for strat, score in results.items():
        print(f"{strat}: {score:.4f}")

    print(
        f"\n🚀 SUCCESS: The optimal modeling strategy chosen is: **{best_strategy}**"
    )


# Execute the Strategy Selector
optimal_modeling_strategy(
    X_train, X_test, y_train, y_test, continuous_features, categorical_features
)


--- Strategy Selection Performance (Adjusted R²) ---
Baseline (Continuous + Categorical): 0.0743
Polynomial Strategy: -0.0009
Interaction Strategy: 0.0459
Full Combined Strategy: 0.0182

🚀 SUCCESS: The optimal modeling strategy chosen is: **Baseline (Continuous + Categorical)**
